# 01 · Masks

Turn the tap points saved on the phone into masks with **SAM 2**, then review them one by one.

1. SAM 2.1 runs once per tapped object, with the tap as a positive point. Of SAM's guesses, the
   one whose size best fits a ~5.5 cm object at the measured distance is used.
2. You step through the samples: **accept**, **reject**, or **edit** the mask and then accept.
3. `labels.png`, `instances.png`, your edits and `mask_status` are written back into each zip.

SAM is used only here, to make training labels. It is never used at inference.

Run this notebook in **Jupyter Lab in the browser** (`jupyter lab notebooks`); the click-to-edit
image needs it. It is quickest to make the masks from a terminal first, with
`python -m unetultra.masks data/*.zip` (add `--redo` to start again from scratch): this notebook
then loads them and goes straight to the review.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
from unetultra.session import find_sessions
from unetultra.validate import validate_session

ZIPS = find_sessions(DATA)          # or list the zips you want: ["../data/session_....zip"]
for path in ZIPS:
    print(validate_session(path).format())

## Load SAM 2

Needed for editing, and for any sample that has no mask yet.

In [ ]:
from unetultra.masks import Sam2Predictor

# CPU is the safe choice (about a second per object). device="mps" has been seen to stall.
predictor = Sam2Predictor("facebook/sam2.1-hiera-small", device="cpu")   # or "facebook/sam2.1-hiera-base-plus"
print("SAM 2 running on", predictor.device)

In [ ]:
import time
from unetultra.masks import MaskReview

review = MaskReview(ZIPS, predictor, object_size_cm=5.5)
started = time.time()
# Only samples without a saved mask are run. One line each, printed before SAM starts on it.
review.run(progress=lambda done, total, sample_id: print(
    f"{done}/{total}  sample {sample_id}  ({time.time() - started:.0f}s so far)", flush=True))
print(f"done in {time.time() - started:.0f}s")
review.counts()

## Review one by one

The picture is zoomed on the tapped object; the mask is tinted with a white edge, and the tap is
the yellow cross.

- **accept & next** / **reject & next**: decide and move on. **back** returns to the previous one.
- If the mask is wrong, pick a mode and click on the picture:
  - **add to mask**: click a part of the case that is missing.
  - **remove from mask**: click something wrongly covered (table, hand).
  - **restart from click**: the tap landed beside the case; click the case and SAM starts again there.
  - **draw outline**: click points around the case, then **close outline**. That shape becomes the mask.
- **other SAM guess** switches between SAM's alternative masks for the same clicks.
- **reset mask** undoes every edit; **zoom in/out** shows the whole photo.

Progress is saved every 10 decisions and when you reach the end, so you can stop and come back:
re-running the notebook continues with the samples still pending.

In [ ]:
%matplotlib widget
from unetultra.review_ui import step_review

panel, stepper = step_review(review, only="pending")   # only=None to go through every sample again
panel

## Save and check

Saves anything not yet written and validates the zips.

In [ ]:
print(review.save())
for path in ZIPS:
    print(validate_session(path).format())

## Overview grid (optional)

All samples at a glance, 12 per page, to spot anything you want to revisit. To redo one, run
`stepper = step_review(review, only=None)` above, or set it back to pending here and re-run the
review cell.

In [ ]:
from unetultra.review_ui import review_grid

review_grid(review, page=0, per_page=12)

## Check against the fake session's known shapes

Only meaningful for zips made by `tools/make_fake_session.py`: the true masks are regenerated
from the same seed and compared with SAM's. The target is Dice > 0.95.

In [ ]:
import tempfile
from unetultra.fake_session import make_fake_session
from unetultra.masks import dice
from unetultra.session import Session

for path in ZIPS:
    with Session(path) as made:
        if made.manifest["app_version"] != "fake":
            continue
        seed = int(made.session_id.removeprefix("fake"))
        size = tuple(made.meta(made.sample_ids[0])["image_size"])
        with tempfile.TemporaryDirectory() as tmp:
            truth_zip = make_fake_session(Path(tmp) / "truth.zip", n_samples=len(made.sample_ids), seed=seed,
                                          split=made.split, classes=made.classes, image_size=size, with_masks=True)
            with Session(truth_zip) as truth:
                scores = [dice(made.instances(s) > 0, truth.instances(s) > 0)
                          for s in made.sample_ids if made.instances(s) is not None and truth.meta(s)["objects"]]
        print(f"{path}: Dice min {min(scores):.3f}, mean {sum(scores) / len(scores):.3f} over {len(scores)} samples")